# Token Transfers EDA

In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

In [ ]:
tk1 = pd.read_csv('../data/token_transfers.csv')

In [ ]:
tk1.shape
print (f'the data has {tk1.shape[0]} rows')

In [ ]:
CONTRACTS = {
    '0xdac17f958d2ee523a2206206994597c13d831ec7': 'USDT',
    '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48': 'USDC', 
    '0x6b175474e89094c44da98b954eedeac495271d0f': 'DAI',
    '0x8e870d67f660d95d5be530380d0ec0bd388289e1': 'PAX',
    '0xa47c8bf37f92abed4a126bda807a7b7498661acd': 'USTC',  
    '0xd2877702675e6ceb975b4a1dff9fb7baf4c91ea9': 'WLUNA', 
}

In [ ]:
tk1['contract_address'].unique()

In [ ]:
tk1['datetime'] = pd.to_datetime(tk1['time_stamp'], unit='s')
# Extract date components
tk1['date'] = tk1['datetime'].dt.date
tk1['year'] = tk1['datetime'].dt.year
tk1['month'] = tk1['datetime'].dt.month
tk1['month_name'] = tk1['datetime'].dt.month_name()
tk1['day'] = tk1['datetime'].dt.day
tk1['hour'] = tk1['datetime'].dt.hour
tk1['minute'] = tk1['datetime'].dt.minute

# View results
print(tk1[['time_stamp', 'datetime', 'date', 'month', 'month_name']].head())
print(tk1[['time_stamp', 'datetime', 'date', 'month', 'month_name']].tail())

In [ ]:
token1= tk1.drop(columns = ['date','year','month','month_name','day','hour','minute'])
token1.head()

In [ ]:
token1['contract_address'] = token1['contract_address'].map(CONTRACTS)

In [ ]:
token1.head()

In [ ]:
token1.tail()

In [ ]:
# focus on USTC
ustc = token1[token1['contract_address'] == 'USTC']

# Hourly volume
hourly = ustc.set_index('datetime').resample('h')['value'].sum()

# Plot
plt.figure(figsize=(14, 8))
plt.plot(hourly.index, hourly.values, linewidth=2, color='red')
plt.title('When Did USTC Panic Start?')
plt.ylabel('Hourly Volume')
plt.xlabel('Date')
plt.grid(True)
plt.show()

# Find the peak
peak_hour = hourly.idxmax()
print(f"Panic peaked at: {peak_hour}")

In [ ]:
WHALE_THRESHOLD = 100000      # $100k+ = Whale
MEDIUM_THRESHOLD = 10000      # $10k-$100k = Medium investor
SMALL_THRESHOLD = 1000        # $1k-$10k = Small investor
                               # <$1k = Retail

token1['investor_type'] = pd.cut(token1['value'], 
                              bins=[0, SMALL_THRESHOLD, MEDIUM_THRESHOLD, WHALE_THRESHOLD, float('inf')],
                              labels=['Retail (<$1k)', 'Small ($1k-$10k)', 
                                     'Medium ($10k-$100k)', 'Whale (>$100k)'])

type_counts = token1['investor_type'].value_counts()
print("\nTransaction breakdown:")
for inv_type, count in type_counts.items():
    pct = (count / len(token1)) * 100
    print(f"  {inv_type}: {count:,} ({pct:.1f}%)")


hourly_by_type = token1.groupby([token1['datetime'].dt.floor('h'), 'investor_type']).size().reset_index(name='count')
hourly_by_type.columns = ['datetime', 'investor_type', 'count']

fig, ax = plt.subplots(figsize=(16, 8))

# Plot each investor type
for inv_type in ['Whale (>$100k)', 'Medium ($10k-$100k)', 'Small ($1k-$10k)', 'Retail (<$1k)']:
    data = hourly_by_type[hourly_by_type['investor_type'] == inv_type]
    ax.plot(data['datetime'], data['count'], 
            linewidth=2.5, label=inv_type, marker='o', markersize=4, alpha=0.8)

# Mark crisis period
crisis_start = token1[token1['value'] > WHALE_THRESHOLD]['datetime'].min()
ax.axvline(crisis_start, color='red', linestyle='--', linewidth=2, alpha=0.7, 
           label='First Whale Exit')

ax.set_title('Who Panicked First? Transaction Volume by Investor Type', 
             fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Date/Time', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Transactions', fontsize=14, fontweight='bold')
ax.legend(fontsize=12, loc='upper left')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
plt.close()

# Pivot for stacked area
pivot = hourly_by_type.pivot(index='datetime', columns='investor_type', values='count').fillna(0)

fig, ax = plt.subplots(figsize=(16, 8))

# Create stacked area plot
ax.stackplot(pivot.index, 
             pivot['Whale (>$100k)'], 
             pivot['Medium ($10k-$100k)'],
             pivot['Small ($1k-$10k)'],
             pivot['Retail (<$1k)'],
             labels=['Whale (>$100k)', 'Medium ($10k-$100k)', 
                    'Small ($1k-$10k)', 'Retail (<$1k)'],
             colors=['#E74C3C', '#F39C12', '#3498DB', '#95A5A6'],
             alpha=0.8)

ax.set_title('Panic Composition', 
             fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Date/Time', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Transactions', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=12)
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
plt.close()




In [ ]:

DATA_FILE = '../data/token_transfers.csv'  
CRISIS_DATE = '2022-05-09'  

sns.set_style("whitegrid")
plt.rcParams['figure.facecolor'] = 'white'

df = pd.read_csv(DATA_FILE)

# Prepare data
df['datetime'] = pd.to_datetime(df['time_stamp'], unit='s')
df['value'] = pd.to_numeric(df['value'], errors='coerce')
df = df[df['value'] > 0].copy()
df['date'] = df['datetime'].dt.date
df['hour'] = df['datetime'].dt.hour

print(f"Loaded {len(df):,} transactions")
print(f"Date range: {df['datetime'].min()} to {df['datetime'].max()}")

# Aggregate by date and hour
heatmap_data = df.groupby(['date', 'hour'])['value'].agg(['sum', 'count']).reset_index()

# Create pivot tables
pivot_sum = heatmap_data.pivot(index='date', columns='hour', values='sum')
pivot_count = heatmap_data.pivot(index='date', columns='hour', values='count')

# Create the main visualization
fig = plt.figure(figsize=(24, 16))

# Subplot 1: Total transaction value (log scale)
ax1 = plt.subplot(3, 1, 1)
sns.heatmap(np.log10(pivot_sum + 1), 
            cmap='YlOrRd', 
            cbar_kws={'label': 'Log10(Total Transaction Value + 1)'},
            linewidths=0.1,
            ax=ax1)
ax1.set_title('LUNA Transaction Volume Heatmap (Log Scale)', 
              fontsize=18, fontweight='bold', pad=20)
ax1.set_xlabel('')
ax1.set_ylabel('Date', fontsize=12)

# Subplot 2: Transaction count
ax2 = plt.subplot(3, 1, 2)
sns.heatmap(pivot_count, 
            cmap='Blues', 
            cbar_kws={'label': 'Number of Transactions'},
            linewidths=0.1,
            ax=ax2)
ax2.set_title('LUNA Transaction Count Heatmap', 
              fontsize=18, fontweight='bold', pad=20)
ax2.set_xlabel('')
ax2.set_ylabel('Date', fontsize=12)

# Subplot 3: Average transaction value
pivot_avg = (pivot_sum / pivot_count).fillna(0)
ax3 = plt.subplot(3, 1, 3)
sns.heatmap(np.log10(pivot_avg + 1), 
            cmap='RdYlGn_r', 
            cbar_kws={'label': 'Log10(Average Transaction Value + 1)'},
            linewidths=0.1,
            ax=ax3)
ax3.set_title('LUNA Average Transaction Value Heatmap (Log Scale)', 
              fontsize=18, fontweight='bold', pad=20)
ax3.set_xlabel('Hour of Day (UTC)', fontsize=12)
ax3.set_ylabel('Date', fontsize=12)

# Add crisis date marker if it's in the data
crisis_dt = pd.to_datetime(CRISIS_DATE).date()
if crisis_dt in pivot_sum.index:
    crisis_idx = list(pivot_sum.index).index(crisis_dt)
    for ax in [ax1, ax2, ax3]:
        ax.axhline(y=crisis_idx + 0.5, color='red', linestyle='--', 
                   linewidth=3, alpha=0.7, label='Crisis Date')
        ax.legend(loc='upper right')

plt.tight_layout()

# Print summary statistics
print(f"Total transactions: {len(df):,}")
print(f"Total value: {df['value'].sum():,.2f}")
print(f"Average value: {df['value'].mean():,.2f}")
print(f"Median value: {df['value'].median():,.2f}")
print(f"Max single transaction: {df['value'].max():,.2f}")
print(f"Peak day (by volume): {df.groupby('date')['value'].sum().idxmax()}")
print(f"Peak hour (by volume): {df.groupby('hour')['value'].sum().idxmax()}:00")

plt.show()

In [ ]:
token2 = token1[token1['datetime'] > '2022-05-09 18:30:00']

In [ ]:
token2[token2['contract_address'] == 'USDT'].shape

In [ ]:
coins = ['USDT','USTC','DAI','PAX','USDC','WLUNA']
total = 0

for coin in coins: 
    print(f'the number of transaction after 2022-05-09 18:30:00 for {coin} is {token2[token2['contract_address'] == coin].shape[0]}.')
    total += token2[token2['contract_address'] == coin].shape[0]

print (total)


In [ ]:
token3 = token1[token1['datetime'] <= '2022-05-09 18:30:00']

In [ ]:
coins = ['USDT','USTC','DAI','PAX','USDC','WLUNA']
sum = 0
for coin in coins: 
    print(f'the number of transaction before 2022-05-09 18:30:00 {coin} is {token3[token3['contract_address'] == coin].shape[0]}.')
    sum += token3[token3['contract_address'] == coin].shape[0] 

print (sum)
    

In [ ]:
CRISIS_DATE = '2022-05-10'

# Aggregate data by date and hour
print("\nAggregating data...")
agg_data = tk1.groupby(['date', 'hour'])['value'].agg(['sum', 'count']).reset_index()

# Create pivot tables
pivot_sum = agg_data.pivot(index='date', columns='hour', values='sum')
pivot_count = agg_data.pivot(index='date', columns='hour', values='count')

print(f"✓ Created heatmap data: {len(pivot_sum)} days x 24 hours")

fig, axes = plt.subplots(2, 1, figsize=(22, 14))

# Heatmap 1: Transaction VALUE (sum)
sns.heatmap(np.log10(pivot_sum + 1), 
            cmap='YlOrRd',
            cbar_kws={'label': 'Log10(Total Transaction Value + 1)'},
            ax=axes[0],
            linewidths=0.1)
axes[0].set_title('USDT: Total Transaction VALUE by Date and Hour', 
                  fontsize=18, fontweight='bold', pad=20)
axes[0].set_xlabel('Hour of Day (UTC)', fontsize=13)
axes[0].set_ylabel('Date', fontsize=13)

# Heatmap 2: Transaction COUNT
sns.heatmap(pivot_count, 
            cmap='Blues',
            cbar_kws={'label': 'Number of Transactions'},
            ax=axes[1],
            linewidths=0.1)
axes[1].set_title('USDT: Transaction COUNT by Date and Hour', 
                  fontsize=18, fontweight='bold', pad=20)
axes[1].set_xlabel('Hour of Day (UTC)', fontsize=13)
axes[1].set_ylabel('Date', fontsize=13)

# Add crisis date marker if it exists in the data
crisis_dt = pd.to_datetime(CRISIS_DATE).date()
if crisis_dt in pivot_sum.index:
    crisis_idx = list(pivot_sum.index).index(crisis_dt)
    for ax in axes:
        ax.axhline(y=crisis_idx + 0.5, color='red', linestyle='--', 
                   linewidth=3, alpha=0.7, label='May 10, 2022 (Crisis)')
        ax.legend(loc='upper right', fontsize=11)

plt.suptitle('USDT Transaction Heatmaps: Value & Count', 
             fontsize=22, fontweight='bold', y=0.995)
plt.tight_layout()


# Print summary stat
plt.show()

## LUNA Analysis

In [ ]:


# LUNA Contract Addresses
LUNA_CONTRACTS = [
    '0xd2877702675e6ceb975b4a1dff9fb7baf4c91ea9',  # Wrapped LUNA (WLUNA) on Ethereum
]

WHALE_THRESHOLD = 100000      # > $100k
MEDIUM_THRESHOLD = 10000      # $10k - $100k
SMALL_THRESHOLD = 1000        # $1k - $10k

CRISIS_DATE = '2022-05-10'
LUNA_DATE = '2022-05-09'


# Find contract column
contract_col = None
for col in tk1.columns:
    if 'contract' in col.lower():
        contract_col = col
        break

if contract_col is None:
    print("\n Could not find contract address column")
    exit()

# Filter for LUNA
tk1[contract_col] = tk1[contract_col].str.lower()
LUNA_CONTRACTS_LOWER = [addr.lower() for addr in LUNA_CONTRACTS]
df_luna = tk1[tk1[contract_col].isin(LUNA_CONTRACTS_LOWER)].copy()

print(f" Found {len(df_luna):,} LUNA transactions")

if len(df_luna) == 0:
    print("\nNo LUNA transactions found!")
    exit()

# Prepare data
df_luna['datetime'] = pd.to_datetime(df_luna['time_stamp'], unit='s')
df_luna['value'] = pd.to_numeric(df_luna['value'], errors='coerce')
df_luna = df_luna[df_luna['value'] > 0].copy()

print(f" Date range: {df_luna['datetime'].min()} to {df_luna['datetime'].max()}")

def categorize_investor(value):
    if value >= WHALE_THRESHOLD:
        return 'Whale (>$100k)'
    elif value >= MEDIUM_THRESHOLD:
        return 'Medium ($10k-$100k)'
    elif value >= SMALL_THRESHOLD:
        return 'Small ($1k-$10k)'
    else:
        return 'Retail (<$1k)'

df_luna['investor_type'] = df_luna['value'].apply(categorize_investor)

# Print distribution
print("\nInvestor Distribution:")
print(df_luna['investor_type'].value_counts().sort_index())

print("\nTotal Value by Investor Type:")
for inv_type in ['Whale (>$100k)', 'Medium ($10k-$100k)', 'Small ($1k-$10k)', 'Retail (<$1k)']:
    total = df_luna[df_luna['investor_type'] == inv_type]['value'].sum()
    pct = total / df_luna['value'].sum() * 100
    print(f"  {inv_type:25} ${total:15,.2f} ({pct:5.1f}%)")


# Aggregate data by hour
hourly_value = df_luna.groupby([df_luna['datetime'].dt.floor('h'), 'investor_type'])['value'].sum().reset_index()
hourly_value.columns = ['datetime', 'investor_type', 'total_value']

hourly_count = df_luna.groupby([df_luna['datetime'].dt.floor('h'), 'investor_type']).size().reset_index(name='count')
hourly_count.columns = ['datetime', 'investor_type', 'count']

# Colors
colors = {
    'Whale (>$100k)': '#e74c3c',
    'Medium ($10k-$100k)': '#f39c12',
    'Small ($1k-$10k)': '#3498db',
    'Retail (<$1k)': '#2ecc71'
}

crisis_dt = pd.to_datetime(CRISIS_DATE)
luna_dt = pd.to_datetime(LUNA_DATE)


fig, axes = plt.subplots(2, 1, figsize=(22, 14))

# Panel 1: VALUE
ax1 = axes[0]
for inv_type in ['Whale (>$100k)', 'Medium ($10k-$100k)', 'Small ($1k-$10k)', 'Retail (<$1k)']:
    data = hourly_value[hourly_value['investor_type'] == inv_type]
    if len(data) > 0:
        ax1.plot(data['datetime'], data['total_value'], 
                linewidth=3.5, label=inv_type, color=colors[inv_type], alpha=0.85)

ax1.axvline(crisis_dt, color='red', linestyle='--', linewidth=3, alpha=0.7, label='May 10, 2022')
ax1.set_title('LUNA Transaction VALUE by Investor Type', fontsize=18, fontweight='bold', pad=15)
ax1.set_ylabel('Total Transaction Value ($)', fontsize=14, fontweight='bold')
ax1.legend(fontsize=12, loc='upper left', framealpha=0.95)
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')
ax1.tick_params(axis='x', labelbottom=False)
ax1.tick_params(axis='both', labelsize=11)

# Panel 2: COUNT
ax2 = axes[1]
for inv_type in ['Whale (>$100k)', 'Medium ($10k-$100k)', 'Small ($1k-$10k)', 'Retail (<$1k)']:
    data = hourly_count[hourly_count['investor_type'] == inv_type]
    if len(data) > 0:
        ax2.plot(data['datetime'], data['count'], 
                linewidth=3.5, label=inv_type, color=colors[inv_type], alpha=0.85)

ax2.axvline(crisis_dt, color='red', linestyle='--', linewidth=3, alpha=0.7)
ax2.set_title('LUNA Transaction COUNT by Investor Type', fontsize=18, fontweight='bold', pad=15)
ax2.set_xlabel('Date/Time', fontsize=14, fontweight='bold')
ax2.set_ylabel('Number of Transactions', fontsize=14, fontweight='bold')
ax2.legend(fontsize=12, loc='upper left', framealpha=0.95)
ax2.grid(True, alpha=0.3)
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45)
ax2.tick_params(axis='both', labelsize=11)

plt.suptitle('LUNA Whale vs Retail: Complete Timeline Analysis', 
             fontsize=24, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()
plt.close()

# ============================================================================
# SUMMARY STATISTICS
# ============================================================================

before = df_luna[df_luna['datetime'] < crisis_dt]
after = df_luna[df_luna['datetime'] >= crisis_dt]

print("\nBEFORE Crisis (Before May 10, 2022):")
for inv_type in ['Whale (>$100k)', 'Medium ($10k-$100k)', 'Small ($1k-$10k)', 'Retail (<$1k)']:
    count = len(before[before['investor_type'] == inv_type])
    total = before[before['investor_type'] == inv_type]['value'].sum()
    if count > 0:
        avg = total / count
        print(f"  {inv_type:25} | Count: {count:6,} | Total: ${total:12,.2f} | Avg: ${avg:10,.2f}")

print("\nAFTER Crisis (After May 10, 2022):")
for inv_type in ['Whale (>$100k)', 'Medium ($10k-$100k)', 'Small ($1k-$10k)', 'Retail (<$1k)']:
    count = len(after[after['investor_type'] == inv_type])
    total = after[after['investor_type'] == inv_type]['value'].sum()
    if count > 0:
        avg = total / count
        print(f"  {inv_type:25} | Count: {count:6,} | Total: ${total:12,.2f} | Avg: ${avg:10,.2f}")

print("\nCHANGE (%):")
for inv_type in ['Whale (>$100k)', 'Medium ($10k-$100k)', 'Small ($1k-$10k)', 'Retail (<$1k)']:
    count_b = len(before[before['investor_type'] == inv_type])
    count_a = len(after[after['investor_type'] == inv_type])
    count_chg = ((count_a - count_b) / count_b * 100) if count_b > 0 else 0
    
    value_b = before[before['investor_type'] == inv_type]['value'].sum()
    value_a = after[after['investor_type'] == inv_type]['value'].sum()
    value_chg = ((value_a - value_b) / value_b * 100) if value_b > 0 else 0
    
    print(f"  {inv_type:25} | Count: {count_chg:+7.1f}% | Value: {value_chg:+7.1f}%")


whale_count_change = ((len(after[after['investor_type'] == 'Whale (>$100k)']) - 
                       len(before[before['investor_type'] == 'Whale (>$100k)'])) / 
                      len(before[before['investor_type'] == 'Whale (>$100k)']) * 100) if len(before[before['investor_type'] == 'Whale (>$100k)']) > 0 else 0

retail_count_change = ((len(after[after['investor_type'] == 'Retail (<$1k)']) - 
                        len(before[before['investor_type'] == 'Retail (<$1k)'])) / 
                       len(before[before['investor_type'] == 'Retail (<$1k)']) * 100) if len(before[before['investor_type'] == 'Retail (<$1k)']) > 0 else 0

print(f"\n1. Whale activity changed by: {whale_count_change:+.1f}%")
print(f"2. Retail activity changed by: {retail_count_change:+.1f}%")

if abs(whale_count_change) > abs(retail_count_change):
    print(f"\n→ WHALES reacted MORE strongly than retail (±{abs(whale_count_change - retail_count_change):.1f}% difference)")
    print("   This suggests institutional/large holders led the panic")
else:
    print(f"\n→ RETAIL reacted MORE strongly than whales (±{abs(retail_count_change - whale_count_change):.1f}% difference)")
    print("   This suggests small holders panicked first")

# Who dominated the volume?
whale_pct = (after[after['investor_type'] == 'Whale (>$100k)']['value'].sum() / 
             after['value'].sum() * 100) if len(after) > 0 else 0

print(f"\n3. After May 10, Whales accounted for {whale_pct:.1f}% of total transaction value")
if whale_pct > 60:
    print("   → Large holders dominated post-crisis trading")
elif whale_pct > 40:
    print("   → Mixed activity from all investor sizes")
else:
    print("   → Retail/small holders dominated post-crisis trading")

plt.show()


In [ ]:

start_date = '2022-04-28'
end_date = '2022-05-25'

fig, axes = plt.subplots(2, 1, figsize=(14, 9))

ax1 = axes[0]

hourly_count_filtered = hourly_count[
    (hourly_count['datetime'] >= start_date) & 
    (hourly_count['datetime'] <= end_date)
]

for inv_type in ['Whale (>$100k)', 'Medium ($10k-$100k)', 'Small ($1k-$10k)', 'Retail (<$1k)']:
    data = hourly_count_filtered[hourly_count_filtered['investor_type'] == inv_type]
    if len(data) > 0:
        ax1.plot(data['datetime'], data['count'], 
                linewidth=3.5, label=inv_type, color=colors[inv_type], alpha=0.85)

ax1.axvline(crisis_dt, color='red', linestyle='--', linewidth=3, alpha=0.7, 
           label='May 10 (UST Collapse)')
ax1.axvline(luna_dt, color='orange', linestyle='--', linewidth=3, alpha=0.7, 
           label='May 9 (LUNA Collapse)')

ax1.set_title('LUNA: Who Panicked First? Transaction Count by Investor Type', 
             fontsize=18, fontweight='bold', pad=15)
ax1.set_ylabel('Number of Transactions', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, loc='upper left', framealpha=0.95)
ax1.grid(True, alpha=0.3, linewidth=0.5)
ax1.tick_params(axis='x', labelbottom=False)  #

ax2 = axes[1]

# Load and prepare WLUNA data with date filter
wluna = pd.read_csv('../data/wluna_price_data.csv')
wluna['datetime'] = pd.to_datetime(wluna['timestamp'], unit='s')

time = wluna[
    (wluna['datetime'] >= start_date) & 
    (wluna['datetime'] <= end_date)
]

ax2.plot(time['datetime'], time['close'], linewidth=2.5, color='darkblue')
ax2.axvline(crisis_dt, color='red', linestyle='--', linewidth=3, alpha=0.7)
ax2.axvline(luna_dt, color='orange', linestyle='--', linewidth=3, alpha=0.7)

ax2.set_title('LUNA Price Collapse', fontsize=18, fontweight='bold', pad=15)
ax2.set_xlabel('Date/Time', fontsize=14, fontweight='bold')
ax2.set_ylabel('Price (USD)', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3, linewidth=0.5)
ax2.tick_params(axis='x', rotation=45)

fig.suptitle(f'LUNA Crisis: Transaction Activity vs Price ({start_date} to {end_date})', 
             fontsize=20, fontweight='bold', y=0.995)

plt.tight_layout()
plt.show()

## USDT analysis


In [ ]:
USDT_CONTRACT = '0xdac17f958d2ee523a2206206994597c13d831ec7'  

# Investor thresholds
WHALE_THRESHOLD = 100000      # > $100k
MEDIUM_THRESHOLD = 10000      # $10k - $100k
SMALL_THRESHOLD = 1000        # $1k - $10k

CRISIS_DATE = '2022-05-10'

contract_col = None
for col in tk1.columns:
    if 'contract' in col.lower() or 'token' in col.lower():
        if col != 'token_name':  
            contract_col = col
            break


tk1[contract_col] = tk1[contract_col].str.lower()
USDT_CONTRACT = USDT_CONTRACT.lower()

# Filter for USDT only
print(f"\nFiltering for USDT contract: {USDT_CONTRACT}")
df_usdt = tk1[tk1[contract_col] == USDT_CONTRACT].copy()

print(f" Found {len(df_usdt):,} USDT transactions ({len(df_usdt)/len(tk1)*100:.2f}% of total)")

if len(df_usdt) == 0:
    print("\n No USDT transactions found!")
    print("\nAvailable contract addresses in your data:")
    print(tk1[contract_col].value_counts().head(10))
    print("\nPlease check and update USDT_CONTRACT in the script")
    exit()


df_usdt['datetime'] = pd.to_datetime(df_usdt['time_stamp'], unit='s')
df_usdt['value'] = pd.to_numeric(df_usdt['value'], errors='coerce')
df_usdt = df_usdt[df_usdt['value'] > 0].copy()

print(f"\n✓ Date range: {df_usdt['datetime'].min()} to {df_usdt['datetime'].max()}")
print(f"✓ Value range: ${df_usdt['value'].min():.2f} to ${df_usdt['value'].max():,.2f}")

def categorize_investor(value):
    if value >= WHALE_THRESHOLD:
        return 'Whale (>$100k)'
    elif value >= MEDIUM_THRESHOLD:
        return 'Medium ($10k-$100k)'
    elif value >= SMALL_THRESHOLD:
        return 'Small ($1k-$10k)'
    else:
        return 'Retail (<$1k)'

df_usdt['investor_type'] = df_usdt['value'].apply(categorize_investor)



# Aggregate data
hourly_value = df_usdt.groupby([df_usdt['datetime'].dt.floor('h'), 'investor_type'])['value'].sum().reset_index()
hourly_value.columns = ['datetime', 'investor_type', 'total_value']

hourly_count = df_usdt.groupby([df_usdt['datetime'].dt.floor('h'), 'investor_type']).size().reset_index(name='count')
hourly_count.columns = ['datetime', 'investor_type', 'count']

# Colors
colors = {
    'Whale (>$100k)': '#e74c3c',
    'Medium ($10k-$100k)': '#f39c12',
    'Small ($1k-$10k)': '#3498db',
    'Retail (<$1k)': '#2ecc71'
}


print("🎯 Creating: Combined Timeline (VALUE + COUNT)...")

fig, axes = plt.subplots(2, 1, figsize=(20, 12))

# Panel 1: VALUE
ax1 = axes[0]
for inv_type in ['Whale (>$100k)', 'Medium ($10k-$100k)', 'Small ($1k-$10k)', 'Retail (<$1k)']:
    data = hourly_value[hourly_value['investor_type'] == inv_type]
    if len(data) > 0:
        ax1.plot(data['datetime'], data['total_value'], 
                linewidth=3, label=inv_type, color=colors[inv_type], alpha=0.8)

ax1.axvline(crisis_dt, color='red', linestyle='--', linewidth=3, alpha=0.7, label='May 10, 2022')
ax1.set_title('Transaction VALUE by Investor Type', fontsize=16, fontweight='bold')
ax1.set_ylabel('Total Transaction Value ($)', fontsize=12, fontweight='bold')
ax1.legend(fontsize=11, loc='upper left', framealpha=0.9)
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')
ax1.tick_params(axis='x', labelbottom=False)

# Panel 2: COUNT
ax2 = axes[1]
for inv_type in ['Whale (>$100k)', 'Medium ($10k-$100k)', 'Small ($1k-$10k)', 'Retail (<$1k)']:
    data = hourly_count[hourly_count['investor_type'] == inv_type]
    if len(data) > 0:
        ax2.plot(data['datetime'], data['count'], 
                linewidth=3, label=inv_type, color=colors[inv_type], alpha=0.8)

ax2.axvline(crisis_dt, color='red', linestyle='--', linewidth=3, alpha=0.7)
ax2.set_title('Transaction COUNT by Investor Type', fontsize=16, fontweight='bold')
ax2.set_xlabel('Date/Time', fontsize=12, fontweight='bold')
ax2.set_ylabel('Number of Transactions', fontsize=12, fontweight='bold')
ax2.legend(fontsize=11, loc='upper left', framealpha=0.9)
ax2.grid(True, alpha=0.3)
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45)

plt.suptitle('USDT Whale vs Retail: Complete Timeline Analysis', 
             fontsize=22, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()
plt.close()

before = df_usdt[df_usdt['datetime'] < crisis_dt]
after = df_usdt[df_usdt['datetime'] >= crisis_dt]

print("\nBEFORE Crisis (Before May 10, 2022):")
for inv_type in ['Whale (>$100k)', 'Medium ($10k-$100k)', 'Small ($1k-$10k)', 'Retail (<$1k)']:
    count = len(before[before['investor_type'] == inv_type])
    total = before[before['investor_type'] == inv_type]['value'].sum()
    print(f"  {inv_type:25} Count: {count:6,} | Value: ${total:15,.2f}")

print("\nAFTER Crisis (After May 10, 2022):")
for inv_type in ['Whale (>$100k)', 'Medium ($10k-$100k)', 'Small ($1k-$10k)', 'Retail (<$1k)']:
    count = len(after[after['investor_type'] == inv_type])
    total = after[after['investor_type'] == inv_type]['value'].sum()
    print(f"  {inv_type:25} Count: {count:6,} | Value: ${total:15,.2f}")

print("\nCHANGE (%):")
for inv_type in ['Whale (>$100k)', 'Medium ($10k-$100k)', 'Small ($1k-$10k)', 'Retail (<$1k)']:
    count_b = len(before[before['investor_type'] == inv_type])
    count_a = len(after[after['investor_type'] == inv_type])
    count_chg = ((count_a - count_b) / count_b * 100) if count_b > 0 else 0
    
    value_b = before[before['investor_type'] == inv_type]['value'].sum()
    value_a = after[after['investor_type'] == inv_type]['value'].sum()
    value_chg = ((value_a - value_b) / value_b * 100) if value_b > 0 else 0
    
    print(f"  {inv_type:25} Count: {count_chg:+7.1f}% | Value: {value_chg:+7.1f}%")

## USDC Analysis

In [ ]:
USDC_CONTRACT = '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48'  


WHALE_THRESHOLD = 100000      # > $100k
MEDIUM_THRESHOLD = 10000      # $10k - $100k
SMALL_THRESHOLD = 1000        # $1k - $10k

CRISIS_DATE = '2022-05-10'

print(f" Loaded {len(tk1):,} total token transfers")
print(f" Columns: {list(tk1.columns)}")

contract_col = None
for col in tk1.columns:
    if 'contract' in col.lower() or 'token' in col.lower():
        if col != 'token_name':
            contract_col = col
            break


tk1[contract_col] = tk1[contract_col].str.lower()
USDC_CONTRACT = USDC_CONTRACT.lower()

# Filter for USDC only
df_usdc = tk1[tk1[contract_col] == USDC_CONTRACT].copy()

if len(df_usdc) == 0:
    print("\n No USDC transactions found!")
    print("\nAvailable contract addresses in your data:")
    print(tk1[contract_col].value_counts().head(10))
    print("\nPlease check and update USDC_CONTRACT in the script")
    exit()

# Prepare datetime and value columns
df_usdc['datetime'] = pd.to_datetime(df_usdc['time_stamp'], unit='s')
df_usdc['value'] = pd.to_numeric(df_usdc['value'], errors='coerce')
df_usdc = df_usdc[df_usdc['value'] > 0].copy()

print(f"\n Date range: {df_usdc['datetime'].min()} to {df_usdc['datetime'].max()}")
print(f" Value range: ${df_usdc['value'].min():.2f} to ${df_usdc['value'].max():,.2f}")

# Categorize investors
def categorize_investor(value):
    if value >= WHALE_THRESHOLD:
        return 'Whale (>$100k)'
    elif value >= MEDIUM_THRESHOLD:
        return 'Medium ($10k-$100k)'
    elif value >= SMALL_THRESHOLD:
        return 'Small ($1k-$10k)'
    else:
        return 'Retail (<$1k)'

df_usdc['investor_type'] = df_usdc['value'].apply(categorize_investor)

# Aggregate data
hourly_value = df_usdc.groupby([df_usdc['datetime'].dt.floor('h'), 'investor_type'])['value'].sum().reset_index()
hourly_value.columns = ['datetime', 'investor_type', 'total_value']

hourly_count = df_usdc.groupby([df_usdc['datetime'].dt.floor('h'), 'investor_type']).size().reset_index(name='count')
hourly_count.columns = ['datetime', 'investor_type', 'count']

# Colors
colors = {
    'Whale (>$100k)': '#e74c3c',
    'Medium ($10k-$100k)': '#f39c12',
    'Small ($1k-$10k)': '#3498db',
    'Retail (<$1k)': '#2ecc71'
}

fig, axes = plt.subplots(2, 1, figsize=(20, 12))

# Panel 1: VALUE
ax1 = axes[0]
for inv_type in ['Whale (>$100k)', 'Medium ($10k-$100k)', 'Small ($1k-$10k)', 'Retail (<$1k)']:
    data = hourly_value[hourly_value['investor_type'] == inv_type]
    if len(data) > 0:
        ax1.plot(data['datetime'], data['total_value'], 
                linewidth=3, label=inv_type, color=colors[inv_type], alpha=0.8)

ax1.axvline(crisis_dt, color='red', linestyle='--', linewidth=3, alpha=0.7, label='May 10, 2022')
ax1.set_title('Transaction VALUE by Investor Type', fontsize=16, fontweight='bold')
ax1.set_ylabel('Total Transaction Value ($)', fontsize=12, fontweight='bold')
ax1.legend(fontsize=11, loc='upper left', framealpha=0.9)
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')
ax1.tick_params(axis='x', labelbottom=False)

# Panel 2: COUNT
ax2 = axes[1]
for inv_type in ['Whale (>$100k)', 'Medium ($10k-$100k)', 'Small ($1k-$10k)', 'Retail (<$1k)']:
    data = hourly_count[hourly_count['investor_type'] == inv_type]
    if len(data) > 0:
        ax2.plot(data['datetime'], data['count'], 
                linewidth=3, label=inv_type, color=colors[inv_type], alpha=0.8)

ax2.axvline(crisis_dt, color='red', linestyle='--', linewidth=3, alpha=0.7)
ax2.set_title('Transaction COUNT by Investor Type', fontsize=16, fontweight='bold')
ax2.set_xlabel('Date/Time', fontsize=12, fontweight='bold')
ax2.set_ylabel('Number of Transactions', fontsize=12, fontweight='bold')
ax2.legend(fontsize=11, loc='upper left', framealpha=0.9)
ax2.grid(True, alpha=0.3)
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45)

plt.suptitle('USDC Whale vs Retail: Complete Timeline Analysis', 
             fontsize=22, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()
plt.close()

before = df_usdc[df_usdc['datetime'] < crisis_dt]
after = df_usdc[df_usdc['datetime'] >= crisis_dt]

print("\nBEFORE Crisis (Before May 10, 2022):")
for inv_type in ['Whale (>$100k)', 'Medium ($10k-$100k)', 'Small ($1k-$10k)', 'Retail (<$1k)']:
    count = len(before[before['investor_type'] == inv_type])
    total = before[before['investor_type'] == inv_type]['value'].sum()
    print(f"  {inv_type:25} Count: {count:6,} | Value: ${total:15,.2f}")

print("\nAFTER Crisis (After May 10, 2022):")
for inv_type in ['Whale (>$100k)', 'Medium ($10k-$100k)', 'Small ($1k-$10k)', 'Retail (<$1k)']:
    count = len(after[after['investor_type'] == inv_type])
    total = after[after['investor_type'] == inv_type]['value'].sum()
    print(f"  {inv_type:25} Count: {count:6,} | Value: ${total:15,.2f}")

print("\nCHANGE (%):")
for inv_type in ['Whale (>$100k)', 'Medium ($10k-$100k)', 'Small ($1k-$10k)', 'Retail (<$1k)']:
    count_b = len(before[before['investor_type'] == inv_type])
    count_a = len(after[after['investor_type'] == inv_type])
    count_chg = ((count_a - count_b) / count_b * 100) if count_b > 0 else 0
    
    value_b = before[before['investor_type'] == inv_type]['value'].sum()
    value_a = after[after['investor_type'] == inv_type]['value'].sum()
    value_chg = ((value_a - value_b) / value_b * 100) if value_b > 0 else 0
    
    print(f"  {inv_type:25} Count: {count_chg:+7.1f}% | Value: {value_chg:+7.1f}%")

In [ ]:
print(" Creating: Transaction COUNT Timeline...")
from datetime import datetime
luna_dt = pd.to_datetime('2022-05-09')

fig, ax = plt.subplots(figsize=(14, 9))

for inv_type in ['Whale (>$100k)', 'Medium ($10k-$100k)', 'Small ($1k-$10k)', 'Retail (<$1k)']:
    data = hourly_count[hourly_count['investor_type'] == inv_type]
    if len(data) > 0:
        ax.plot(data['datetime'], data['count'], 
                linewidth=3, label=inv_type, color=colors[inv_type], alpha=0.8)

ax.axvline(crisis_dt, color='red', linestyle='--', linewidth=3, alpha=0.7, 
           label='May 10, 2022 (UST Collapse)')
ax.axvline(luna_dt, color='orange', linestyle='--', linewidth=3, alpha=0.7, 
           label='May 10, 2022 (UST Collapse)')

ax.set_title('USDC: Secondary Destination - Quality Flight', 
             fontsize=20, fontweight='bold', pad=20)
ax.set_xlabel('Date/Time', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Transactions per hour', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 5000)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
plt.close()


## DAI Analysis


In [ ]:
DAI_CONTRACT = '0x6b175474e89094c44da98b954eedeac495271d0f'  

# Investor thresholds
WHALE_THRESHOLD = 100000      # > $100k
MEDIUM_THRESHOLD = 10000      # $10k - $100k
SMALL_THRESHOLD = 1000        # $1k - $10k

CRISIS_DATE = '2022-05-10'


contract_col = None
for col in tk1.columns:
    if 'contract' in col.lower():
        contract_col = col
        break

if contract_col is None:
    print("\n Could not find contract address column")
    exit()

# Filter for DAI
tk1[contract_col] = tk1[contract_col].str.lower()
DAI_CONTRACT = DAI_CONTRACT.lower()
df_dai = tk1[tk1[contract_col] == DAI_CONTRACT].copy()

df_dai['datetime'] = pd.to_datetime(df_dai['time_stamp'], unit='s')
df_dai['value'] = pd.to_numeric(df_dai['value'], errors='coerce')
df_dai = df_dai[df_dai['value'] > 0].copy()

print(f" Date range: {df_dai['datetime'].min()} to {df_dai['datetime'].max()}")

# Categorize investors
def categorize_investor(value):
    if value >= WHALE_THRESHOLD:
        return 'Whale (>$100k)'
    elif value >= MEDIUM_THRESHOLD:
        return 'Medium ($10k-$100k)'
    elif value >= SMALL_THRESHOLD:
        return 'Small ($1k-$10k)'
    else:
        return 'Retail (<$1k)'

df_dai['investor_type'] = df_dai['value'].apply(categorize_investor)

# Aggregate data by hour
hourly_value = df_dai.groupby([df_dai['datetime'].dt.floor('h'), 'investor_type'])['value'].sum().reset_index()
hourly_value.columns = ['datetime', 'investor_type', 'total_value']

hourly_count = df_dai.groupby([df_dai['datetime'].dt.floor('h'), 'investor_type']).size().reset_index(name='count')
hourly_count.columns = ['datetime', 'investor_type', 'count']

# Colors
colors = {
    'Whale (>$100k)': '#e74c3c',
    'Medium ($10k-$100k)': '#f39c12',
    'Small ($1k-$10k)': '#3498db',
    'Retail (<$1k)': '#2ecc71'
}

crisis_dt = pd.to_datetime(CRISIS_DATE)

fig, axes = plt.subplots(2, 1, figsize=(22, 14))

# Panel 1: VALUE
ax1 = axes[0]
for inv_type in ['Whale (>$100k)', 'Medium ($10k-$100k)', 'Small ($1k-$10k)', 'Retail (<$1k)']:
    data = hourly_value[hourly_value['investor_type'] == inv_type]
    if len(data) > 0:
        ax1.plot(data['datetime'], data['total_value'], 
                linewidth=3.5, label=inv_type, color=colors[inv_type], alpha=0.85)

ax1.axvline(crisis_dt, color='red', linestyle='--', linewidth=3, alpha=0.7, label='May 10, 2022')
ax1.set_title('DAI Transaction VALUE by Investor Type', fontsize=18, fontweight='bold', pad=15)
ax1.set_ylabel('Total Transaction Value ($)', fontsize=14, fontweight='bold')
ax1.legend(fontsize=12, loc='upper left', framealpha=0.95)
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')
ax1.tick_params(axis='x', labelbottom=False)
ax1.tick_params(axis='both', labelsize=11)

# Panel 2: COUNT
ax2 = axes[1]
for inv_type in ['Whale (>$100k)', 'Medium ($10k-$100k)', 'Small ($1k-$10k)', 'Retail (<$1k)']:
    data = hourly_count[hourly_count['investor_type'] == inv_type]
    if len(data) > 0:
        ax2.plot(data['datetime'], data['count'], 
                linewidth=3.5, label=inv_type, color=colors[inv_type], alpha=0.85)

ax2.axvline(crisis_dt, color='red', linestyle='--', linewidth=3, alpha=0.7)
ax2.set_title('DAI Transaction COUNT by Investor Type', fontsize=18, fontweight='bold', pad=15)
ax2.set_xlabel('Date/Time', fontsize=14, fontweight='bold')
ax2.set_ylabel('Number of Transactions', fontsize=14, fontweight='bold')
ax2.legend(fontsize=12, loc='upper left', framealpha=0.95)
ax2.grid(True, alpha=0.3)
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45)
ax2.tick_params(axis='both', labelsize=11)

plt.suptitle('DAI Whale vs Retail: Complete Timeline Analysis', 
             fontsize=24, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()
plt.close()

before = df_dai[df_dai['datetime'] < crisis_dt]
after = df_dai[df_dai['datetime'] >= crisis_dt]

print("\nBEFORE Crisis (Before May 10, 2022):")
for inv_type in ['Whale (>$100k)', 'Medium ($10k-$100k)', 'Small ($1k-$10k)', 'Retail (<$1k)']:
    count = len(before[before['investor_type'] == inv_type])
    total = before[before['investor_type'] == inv_type]['value'].sum()
    if count > 0:
        avg = total / count
        print(f"  {inv_type:25} | Count: {count:6,} | Total: ${total:12,.2f} | Avg: ${avg:10,.2f}")

print("\nAFTER Crisis (After May 10, 2022):")
for inv_type in ['Whale (>$100k)', 'Medium ($10k-$100k)', 'Small ($1k-$10k)', 'Retail (<$1k)']:
    count = len(after[after['investor_type'] == inv_type])
    total = after[after['investor_type'] == inv_type]['value'].sum()
    if count > 0:
        avg = total / count
        print(f"  {inv_type:25} | Count: {count:6,} | Total: ${total:12,.2f} | Avg: ${avg:10,.2f}")

print("\nCHANGE (%):")
for inv_type in ['Whale (>$100k)', 'Medium ($10k-$100k)', 'Small ($1k-$10k)', 'Retail (<$1k)']:
    count_b = len(before[before['investor_type'] == inv_type])
    count_a = len(after[after['investor_type'] == inv_type])
    count_chg = ((count_a - count_b) / count_b * 100) if count_b > 0 else 0
    
    value_b = before[before['investor_type'] == inv_type]['value'].sum()
    value_a = after[after['investor_type'] == inv_type]['value'].sum()
    value_chg = ((value_a - value_b) / value_b * 100) if value_b > 0 else 0
    
    print(f"  {inv_type:25} | Count: {count_chg:+7.1f}% | Value: {value_chg:+7.1f}%")





## PAX Analysis


In [ ]:
PAX_CONTRACT = '0x8e870d67f660d95d5be530380d0ec0bd388289e1'  

# Investor thresholds
WHALE_THRESHOLD = 100000      # > $100k
MEDIUM_THRESHOLD = 10000      # $10k - $100k
SMALL_THRESHOLD = 1000        # $1k - $10k

CRISIS_DATE = '2022-05-10'

contract_col = None
for col in tk1.columns:
    if 'contract' in col.lower():
        contract_col = col
        break

if contract_col is None:
    print("\n Could not find contract address column")
    exit()

# Filter for PAX
tk1[contract_col] = tk1[contract_col].str.lower()
PAX_CONTRACT = PAX_CONTRACT.lower()
df_pax = tk1[tk1[contract_col] == PAX_CONTRACT].copy()


# Prepare data
df_pax['datetime'] = pd.to_datetime(df_pax['time_stamp'], unit='s')
df_pax['value'] = pd.to_numeric(df_pax['value'], errors='coerce')
df_pax = df_pax[df_pax['value'] > 0].copy()

# Categorize investors
def categorize_investor(value):
    if value >= WHALE_THRESHOLD:
        return 'Whale (>$100k)'
    elif value >= MEDIUM_THRESHOLD:
        return 'Medium ($10k-$100k)'
    elif value >= SMALL_THRESHOLD:
        return 'Small ($1k-$10k)'
    else:
        return 'Retail (<$1k)'

df_pax['investor_type'] = df_pax['value'].apply(categorize_investor)


# Aggregate data by hour
hourly_value = df_pax.groupby([df_pax['datetime'].dt.floor('h'), 'investor_type'])['value'].sum().reset_index()
hourly_value.columns = ['datetime', 'investor_type', 'total_value']

hourly_count = df_pax.groupby([df_pax['datetime'].dt.floor('h'), 'investor_type']).size().reset_index(name='count')
hourly_count.columns = ['datetime', 'investor_type', 'count']

# Colors
colors = {
    'Whale (>$100k)': '#e74c3c',
    'Medium ($10k-$100k)': '#f39c12',
    'Small ($1k-$10k)': '#3498db',
    'Retail (<$1k)': '#2ecc71'
}

crisis_dt = pd.to_datetime(CRISIS_DATE)

fig, axes = plt.subplots(2, 1, figsize=(22, 14))

# Panel 1: VALUE
ax1 = axes[0]
for inv_type in ['Whale (>$100k)', 'Medium ($10k-$100k)', 'Small ($1k-$10k)', 'Retail (<$1k)']:
    data = hourly_value[hourly_value['investor_type'] == inv_type]
    if len(data) > 0:
        ax1.plot(data['datetime'], data['total_value'], 
                linewidth=3.5, label=inv_type, color=colors[inv_type], alpha=0.85)

ax1.axvline(crisis_dt, color='red', linestyle='--', linewidth=3, alpha=0.7, label='May 10, 2022')
ax1.set_title('PAX Transaction VALUE by Investor Type', fontsize=18, fontweight='bold', pad=15)
ax1.set_ylabel('Total Transaction Value ($)', fontsize=14, fontweight='bold')
ax1.legend(fontsize=12, loc='upper left', framealpha=0.95)
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')
ax1.tick_params(axis='x', labelbottom=False)
ax1.tick_params(axis='both', labelsize=11)

# Panel 2: COUNT
ax2 = axes[1]
for inv_type in ['Whale (>$100k)', 'Medium ($10k-$100k)', 'Small ($1k-$10k)', 'Retail (<$1k)']:
    data = hourly_count[hourly_count['investor_type'] == inv_type]
    if len(data) > 0:
        ax2.plot(data['datetime'], data['count'], 
                linewidth=3.5, label=inv_type, color=colors[inv_type], alpha=0.85)

ax2.axvline(crisis_dt, color='red', linestyle='--', linewidth=3, alpha=0.7)
ax2.set_title('PAX Transaction COUNT by Investor Type', fontsize=18, fontweight='bold', pad=15)
ax2.set_xlabel('Date/Time', fontsize=14, fontweight='bold')
ax2.set_ylabel('Number of Transactions', fontsize=14, fontweight='bold')
ax2.legend(fontsize=12, loc='upper left', framealpha=0.95)
ax2.grid(True, alpha=0.3)
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45)
ax2.tick_params(axis='both', labelsize=11)

plt.suptitle('PAX Whale vs Retail: Complete Timeline Analysis', 
             fontsize=24, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()
plt.close()


before = df_pax[df_pax['datetime'] < crisis_dt]
after = df_pax[df_pax['datetime'] >= crisis_dt]

print("\nBEFORE Crisis (Before May 10, 2022):")
for inv_type in ['Whale (>$100k)', 'Medium ($10k-$100k)', 'Small ($1k-$10k)', 'Retail (<$1k)']:
    count = len(before[before['investor_type'] == inv_type])
    total = before[before['investor_type'] == inv_type]['value'].sum()
    if count > 0:
        avg = total / count
        print(f"  {inv_type:25} | Count: {count:6,} | Total: ${total:12,.2f} | Avg: ${avg:10,.2f}")

print("\nAFTER Crisis (After May 10, 2022):")
for inv_type in ['Whale (>$100k)', 'Medium ($10k-$100k)', 'Small ($1k-$10k)', 'Retail (<$1k)']:
    count = len(after[after['investor_type'] == inv_type])
    total = after[after['investor_type'] == inv_type]['value'].sum()
    if count > 0:
        avg = total / count
        print(f"  {inv_type:25} | Count: {count:6,} | Total: ${total:12,.2f} | Avg: ${avg:10,.2f}")

print("\nCHANGE (%):")
for inv_type in ['Whale (>$100k)', 'Medium ($10k-$100k)', 'Small ($1k-$10k)', 'Retail (<$1k)']:
    count_b = len(before[before['investor_type'] == inv_type])
    count_a = len(after[after['investor_type'] == inv_type])
    count_chg = ((count_a - count_b) / count_b * 100) if count_b > 0 else 0
    
    value_b = before[before['investor_type'] == inv_type]['value'].sum()
    value_a = after[after['investor_type'] == inv_type]['value'].sum()
    value_chg = ((value_a - value_b) / value_b * 100) if value_b > 0 else 0
    
    if count_b > 0:
        print(f"  {inv_type:25} | Count: {count_chg:+7.1f}% | Value: {value_chg:+7.1f}%")


## 4 pane analysis

In [ ]:
CONTRACTS = {
    'DAI': '0x6b175474e89094c44da98b954eedeac495271d0f',
    'USDT': '0xdac17f958d2ee523a2206206994597c13d831ec7',
    'USDC': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48',
    'PAX': '0x8e870d67f660d95d5be530380d0ec0bd388289e1'
}

# Investor thresholds
WHALE_THRESHOLD = 100000      # > $100k
MEDIUM_THRESHOLD = 10000      # $10k - $100k
SMALL_THRESHOLD = 1000        # $1k - $10k

# Key dates
CRISIS_DATES = {
    'LUNA_COLLAPSE': '2022-05-09',
    'UST_COLLAPSE': '2022-05-10'
}


def categorize_investor(value):
    """Categorize investor by transaction size"""
    if value >= WHALE_THRESHOLD:
        return 'Whale (>$100k)'
    elif value >= MEDIUM_THRESHOLD:
        return 'Medium ($10k-$100k)'
    elif value >= SMALL_THRESHOLD:
        return 'Small ($1k-$10k)'
    else:
        return 'Retail (<$1k)'

# Find contract column
contract_col = None
for col in tk1.columns:
    if 'contract' in col.lower():
        contract_col = col
        break

if contract_col is None:
    print("\n Could not find contract address column")
    exit()

# Normalize contract addresses
tk1[contract_col] = tk1[contract_col].str.lower()

# Process each stablecoin
stablecoin_counts = {}
for coin_name, contract_address in CONTRACTS.items():
    data = process_coin_counts(tk1, contract_address, coin_name, contract_col)
    if data is not None:
        stablecoin_counts[coin_name] = data

# Colors for investor types
colors = {
    'Whale (>$100k)': '#e74c3c',
    'Medium ($10k-$100k)': '#f39c12',
    'Small ($1k-$10k)': '#3498db',
    'Retail (<$1k)': '#2ecc71'
}

# Crisis dates
luna_collapse = pd.to_datetime(CRISIS_DATES['LUNA_COLLAPSE'])
ust_collapse = pd.to_datetime(CRISIS_DATES['UST_COLLAPSE'])

# Create figure with 2x2 subplots
fig, axes = plt.subplots(2, 2, figsize=(22, 14))

# Layout: [USDT, USDC]
#         [DAI,  PAX]
plot_positions = {
    'USDT': (0, 0),
    'USDC': (0, 1),
    'DAI': (1, 0),
    'PAX': (1, 1)
}

titles = {
    'USDT': 'USDT: Primary Destination - High Transaction Frequency',
    'USDC': 'USDC: Secondary Destination - Quality Flight',
    'DAI': 'DAI:  Tertiary Destination - DeFi Loyal',
    'PAX': 'PAX: Non-Destination - Minimal Activity Change'
}

# Plot each stablecoin
for coin_name, (row, col) in plot_positions.items():
    ax = axes[row, col]
    
    if coin_name not in stablecoin_counts:
        ax.text(0.5, 0.5, f'{coin_name}\nNo Data Available', 
                ha='center', va='center', fontsize=16, 
                transform=ax.transAxes)
        ax.set_title(titles[coin_name], fontsize=14, fontweight='bold')
        continue
    
    data = stablecoin_counts[coin_name]
    
    # Plot each investor type
    for inv_type in ['Whale (>$100k)', 'Medium ($10k-$100k)', 
                     'Small ($1k-$10k)', 'Retail (<$1k)']:
        inv_data = data[data['investor_type'] == inv_type]
        if len(inv_data) > 0:
            ax.plot(inv_data['datetime'], inv_data['count'],
                   linewidth=3, label=inv_type, 
                   color=colors[inv_type], alpha=0.85)
    
    # Add crisis date markers
    ax.axvline(luna_collapse, color='orange', linestyle='--', 
              linewidth=2.5, alpha=0.7, label='May 9 (LUNA)')
    ax.axvline(ust_collapse, color='red', linestyle='--', 
              linewidth=2.5, alpha=0.7, label='May 10 (UST)')
    
    if coin_name == "USDC":
        ax.set_ylim(0,5000)
    
    # Styling
    ax.set_title(titles[coin_name], fontsize=14, fontweight='bold', pad=10)
    ax.set_xlabel('Date/Time', fontsize=12, fontweight='bold')
    ax.set_ylabel('Number of Transactions per Hour', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, linewidth=0.5)
    ax.tick_params(axis='x', rotation=45, labelsize=10)
    ax.tick_params(axis='y', labelsize=10)
    
    # Add legend only on first plot
    if row == 0 and col == 0:
        ax.legend(fontsize=10, loc='upper left', framealpha=0.95, ncol=1)
    
    # Optional: Add text annotation showing peak count
    

# Overall title
fig.suptitle('Stablecoin Transaction Frequency During Terra-Luna Collapse (May 2022)\nTransaction Count by Investor Type', 
             fontsize=18, fontweight='bold', y=0.995)

# Adjust layout
plt.tight_layout(rect=[0, 0, 1, 0.985])


plt.show()

for coin_name in ['USDT', 'USDC', 'DAI', 'PAX']:
    if coin_name in stablecoin_counts:
        data = stablecoin_counts[coin_name]
        
        # Calculate before/after crisis counts
        before_crisis = data[data['datetime'] < ust_collapse]['count'].sum()
        after_crisis = data[data['datetime'] >= ust_collapse]['count'].sum()
        increase = ((after_crisis - before_crisis) / before_crisis * 100) if before_crisis > 0 else 0
        
        print(f"\n{coin_name}:")
        print(f"  Before May 10: {before_crisis:,} transactions")
        print(f"  After May 10:  {after_crisis:,} transactions")
        print(f"  Change: {increase:+.1f}%")
        

In [ ]:
CONTRACTS = {
    'USDT': '0xdac17f958d2ee523a2206206994597c13d831ec7',
    'USDC': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48'
}

WHALE_THRESHOLD = 100000      # > $100k
MEDIUM_THRESHOLD = 10000      # $10k - $100k
SMALL_THRESHOLD = 1000        # $1k - $10k

# Key dates
CRISIS_DATES = {
    'LUNA_COLLAPSE': '2022-05-09',
    'UST_COLLAPSE': '2022-05-10'
}

def categorize_investor(value):
    """Categorize investor by transaction size"""
    if value >= WHALE_THRESHOLD:
        return 'Whale (>$100k)'
    elif value >= MEDIUM_THRESHOLD:
        return 'Medium ($10k-$100k)'
    elif value >= SMALL_THRESHOLD:
        return 'Small ($1k-$10k)'
    else:
        return 'Retail (<$1k)'

# Find contract column
contract_col = None
for col in tk1.columns:
    if 'contract' in col.lower():
        contract_col = col
        break

if contract_col is None:
    print("\n Could not find contract address column")
    exit()

# Normalize contract addresses
tk1[contract_col] = tk1[contract_col].str.lower()

# Colors for investor types
colors = {
    'Whale (>$100k)': '#e74c3c',
    'Medium ($10k-$100k)': '#f39c12',
    'Small ($1k-$10k)': '#3498db',
    'Retail (<$1k)': '#2ecc71'
}

# Crisis dates
luna_collapse = pd.to_datetime(CRISIS_DATES['LUNA_COLLAPSE'])
ust_collapse = pd.to_datetime(CRISIS_DATES['UST_COLLAPSE'])

# Create figure with 1x2 subplots (side by side)
fig, axes = plt.subplots(2,1 ,figsize=(16, 12))

# Layout: [USDT] [USDC]
plot_positions = {
    'USDT': 0,
    'USDC': 1
}

titles = {
    'USDT': 'USDT: Primary Destination\nHigh Transaction Frequency',
    'USDC': 'USDC: Secondary Destination\nQuality Flight'
}

# Plot each stablecoin
for coin_name, col in plot_positions.items():
    ax = axes[col]
    
    if coin_name not in stablecoin_counts:
        ax.text(0.5, 0.5, f'{coin_name}\nNo Data Available', 
                ha='center', va='center', fontsize=16, 
                transform=ax.transAxes)
        ax.set_title(titles[coin_name], fontsize=14, fontweight='bold')
        continue
    
    data = stablecoin_counts[coin_name]
    
    # Plot each investor type
    for inv_type in ['Whale (>$100k)', 'Medium ($10k-$100k)', 
                     'Small ($1k-$10k)', 'Retail (<$1k)']:
        inv_data = data[data['investor_type'] == inv_type]
        if len(inv_data) > 0:
            ax.plot(inv_data['datetime'], inv_data['count'],
                   linewidth=3, label=inv_type, 
                   color=colors[inv_type], alpha=0.85)
    
    # Add crisis date markers
    ax.axvline(luna_collapse, color='orange', linestyle='--', 
              linewidth=2.5, alpha=0.7, label='May 9 (LUNA)')
    ax.axvline(ust_collapse, color='red', linestyle='--', 
              linewidth=2.5, alpha=0.7, label='May 10 (UST)')
    
    # Set y-limit for USDC
    if coin_name == "USDC":
        ax.set_ylim(0, 5000)
    
    # Styling
    ax.set_title(titles[coin_name], fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel('Date/Time', fontsize=13, fontweight='bold')
    ax.set_ylabel('Number of Transactions per Hour', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, linewidth=0.5)
    ax.tick_params(axis='x', rotation=45, labelsize=11)
    ax.tick_params(axis='y', labelsize=11)
    
    # Add legend on both plots
    ax.legend(fontsize=10, loc='upper left', framealpha=0.95)
    
    print(f"  ✓ Plotted {coin_name}")

# Overall title
fig.suptitle('Stablecoin Transaction Frequency During Terra-Luna Collapse (May 2022)\nTransaction Count by Investor Type', 
             fontsize=18, fontweight='bold', y=0.98)

# Adjust layout
plt.tight_layout(rect=[0, 0, 1, 0.96])

plt.show()

In [ ]:
CONTRACTS = {
    'DAI': '0x6b175474e89094c44da98b954eedeac495271d0f',
    'USDT': '0xdac17f958d2ee523a2206206994597c13d831ec7',
    'USDC': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48',
    'PAX': '0x8e870d67f660d95d5be530380d0ec0bd388289e1'
}

# Investor thresholds
WHALE_THRESHOLD = 100000      # > $100k
MEDIUM_THRESHOLD = 10000      # $10k - $100k
SMALL_THRESHOLD = 1000        # $1k - $10k

# Key dates
CRISIS_DATES = {
    'LUNA_COLLAPSE': '2022-05-09',
    'UST_COLLAPSE': '2022-05-10'
}

# Find contract column
contract_col = None
for col in tk1.columns:
    if 'contract' in col.lower():
        contract_col = col
        break

# Normalize contract addresses
tk1[contract_col] = tk1[contract_col].str.lower()

# Colors for investor types
colors = {
    'Whale (>$100k)': '#e74c3c',
    'Medium ($10k-$100k)': '#f39c12',
    'Small ($1k-$10k)': '#3498db',
    'Retail (<$1k)': '#2ecc71'
}

# Crisis dates
luna_collapse = pd.to_datetime(CRISIS_DATES['LUNA_COLLAPSE'])
ust_collapse = pd.to_datetime(CRISIS_DATES['UST_COLLAPSE'])

# Create figure with 2x2 subplots
fig, axes = plt.subplots(2, 2, figsize=(28, 16))

# Layout: [USDT, USDC]
#         [DAI,  PAX]
plot_positions = {
    'USDT': (0, 0),
    'USDC': (0, 1),
    'DAI': (1, 0),
    'PAX': (1, 1)
}

titles = {
    'USDT': 'USDT: Primary Safe Haven\n(Flight to Liquidity) ~$1B/hour peak',
    'USDC': 'USDC: Quality Safe Haven\n(Flight to Regulation) ~$100M/hour peak',
    'DAI': 'DAI: DeFi Principles Haven\n(Decentralization Preference) ~$10M/hour peak',
    'PAX': 'PAX: Not a Destination\n(Too Small Despite Regulation) ~$1M/hour'
}

# Plot each stablecoin
for coin_name, (row, col) in plot_positions.items():
    ax = axes[row, col]
    
    if coin_name not in stablecoin_data:
        ax.text(0.5, 0.5, f'{coin_name}\nNo Data Available', 
                ha='center', va='center', fontsize=16, 
                transform=ax.transAxes)
        ax.set_title(titles[coin_name], fontsize=14, fontweight='bold')
        continue
    
    data = stablecoin_data[coin_name]
    
    # Plot each investor type
    for inv_type in ['Whale (>$100k)', 'Medium ($10k-$100k)', 
                     'Small ($1k-$10k)', 'Retail (<$1k)']:
        inv_data = data[data['investor_type'] == inv_type]
        if len(inv_data) > 0:
            ax.plot(inv_data['datetime'], inv_data['total_value'],
                   linewidth=2.5, label=inv_type, 
                   color=colors[inv_type], alpha=0.85)
    
    # Add crisis date markers
    ax.axvline(luna_collapse, color='orange', linestyle='--', 
              linewidth=2, alpha=0.7, label='May 9 (LUNA Collapse)')
    ax.axvline(ust_collapse, color='red', linestyle='--', 
              linewidth=2, alpha=0.7, label='May 10 (UST Collapse)')
    
    # Styling
    ax.set_title(titles[coin_name], fontsize=14, fontweight='bold', pad=10)
    ax.set_xlabel('Date/Time', fontsize=12, fontweight='bold')
    ax.set_ylabel('Hourly Transaction Value (USD)', fontsize=12, fontweight='bold')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3, linewidth=0.5)
    ax.tick_params(axis='x', rotation=45, labelsize=10)
    ax.tick_params(axis='y', labelsize=10)
    
    # Only show legend on first plot
    if row == 0 and col == 0:
        ax.legend(fontsize=9, loc='upper left', framealpha=0.95, ncol=1)

# Overall title
fig.suptitle('Stablecoin Capital Flight During Terra-Luna Collapse (May 2022)\nTransaction Value by Investor Type', 
             fontsize=18, fontweight='bold', y=0.995)

# Adjust layout
plt.tight_layout(rect=[0, 0, 1, 0.985])

plt.show()



# DONE